# Sobrevivência das empresas brasileiras (CNPJ)

Fonte: [Cadastro Nacional da Pessoa Jurídica, Receita Federal](https://www.gov.br/receitafederal/pt-br/acesso-a-informacao/dados-abertos/cadastros) · [layout oficial dos arquivos](https://www.gov.br/receitafederal/dados/cnpj-metadados.pdf)

**Como usar**

1. Execute as células em ordem. A seção 2 testa o download com um arquivo minúsculo.
2. Primeiro rode tudo com `PARTES = [1]` (342 MB), só para testar o código.
3. Se deu certo, troque para `PARTES = list(range(10))` (cerca de 5,4 GB) e rode de novo a partir da seção 1.

## 1. Configuração

In [ ]:
import subprocess
import tempfile
import urllib.request
import zipfile
from base64 import b64encode
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap

PARTES = [1]                  # teste; para o resultado final: list(range(10))
MES = "2026-10"               # foto mensal do cadastro
ULTIMO_ANO_COMPLETO = 2025    # último ano inteiro coberto pela foto
COORTES = range(2010, 2025)   # anos de abertura analisados (2010 a 2024)

# Códigos de situação: 1 nula, 2 ativa, 3 suspensa, 4 inapta, 8 baixada
FECHADAS = {4, 8}             # o que conta como "fechou"; compare com {8} na seção 5

# Entidades que têm CNPJ mas não são empresas comuns
DIVISOES_FORA = {84, 94, 97, 99}   # governo, associações/partidos/igrejas, domésticos, organismos
CLASSES_FORA = {8112500}           # condomínios prediais

# Os zips e os resumos de cada parte ficam numa pasta temporária, fora do repositório e do
# OneDrive. Se mudar COORTES ou as exclusões acima, apague os "resumo_*" e "descartes_*" dela.
PASTA_ZIP = Path(tempfile.gettempdir()) / "cnpj"
PASTA_SAIDA = Path("dados")
PASTA_ZIP.mkdir(parents=True, exist_ok=True)
PASTA_SAIDA.mkdir(exist_ok=True)

## 2. Teste de download

A pasta é pública: o "usuário" abaixo é só o código do link de compartilhamento da Receita. Se esta célula falhar no Colab, o servidor pode estar bloqueando o acesso. Nesse caso, rode o notebook no seu computador.

In [ ]:
BASE = f"https://arquivos.receitafederal.gov.br/public.php/webdav/{MES}"
TOKEN = "YggdBLfdninEJX9"


def tamanho_remoto(nome):
    """Tamanho do arquivo no servidor, em bytes, sem baixá-lo."""
    req = urllib.request.Request(f"{BASE}/{nome}", method="HEAD")
    req.add_header("Authorization", "Basic " + b64encode(f"{TOKEN}:".encode()).decode())
    req.add_header("X-Requested-With", "XMLHttpRequest")
    with urllib.request.urlopen(req, timeout=60) as resposta:
        return int(resposta.headers["Content-Length"])


def baixar(nome, tentativas=4):
    """Baixa o arquivo, continuando de onde parou se a conexão cair."""
    destino = PASTA_ZIP / nome
    esperado = tamanho_remoto(nome)
    for _ in range(tentativas):
        atual = destino.stat().st_size if destino.exists() else 0
        if atual == esperado:
            return destino
        if atual > esperado:
            destino.unlink()
        subprocess.run([
            "curl", "-sSLf", "-C", "-", "--retry", "3", "--connect-timeout", "30",
            "--speed-limit", "1000", "--speed-time", "60",
            "-u", f"{TOKEN}:", "-H", "X-Requested-With: XMLHttpRequest",
            "-o", str(destino), f"{BASE}/{nome}",
        ])
    if destino.exists() and destino.stat().st_size == esperado:
        return destino
    raise RuntimeError(f"Não consegui baixar {nome}")


teste = baixar("Cnaes.zip")
print("Download funcionando:", zipfile.ZipFile(teste).namelist())

## 3. Ler e resumir os dados

Cada arquivo é lido em pedaços de 1 milhão de linhas e só 5 das 30 colunas são usadas (nenhuma com nome, telefone ou e-mail). De cada pedaço guardamos apenas contagens, então a memória usada é pequena. O zip é apagado depois de processado.

Truque das datas: elas vêm no formato `AAAAMMDD`. Subtraindo as duas como números e dividindo por 10000 (divisão inteira), obtemos os **anos completos de vida**. Exemplo: `20181120 - 20150701 = 30419`, e `30419 // 10000 = 3` anos.

In [ ]:
COLUNAS = {3: "matriz", 5: "situacao", 6: "data_situacao", 10: "data_abertura", 11: "cnae"}
CHAVES = ["ano_abertura", "divisao", "situacao", "ano_situacao", "anos_de_vida"]


def resumir(pedaco, descartes):
    """Transforma um pedaço do arquivo em contagens por grupo."""
    d = pedaco.rename(columns=COLUNAS).apply(pd.to_numeric, errors="coerce")
    descartes["linhas lidas"] += len(d)

    d = d[d["matriz"] == 1]
    incompleta = d[["situacao", "data_abertura", "cnae"]].isna().any(axis=1)
    descartes["campos essenciais ausentes"] += int(incompleta.sum())
    d = d[~incompleta].copy()
    d["ano_abertura"] = d["data_abertura"] // 10000
    d["divisao"] = d["cnae"] // 100000
    d = d[d["ano_abertura"].isin(COORTES)]

    nao_empresa = d["divisao"].isin(DIVISOES_FORA) | d["cnae"].isin(CLASSES_FORA)
    descartes["não são empresas comuns"] += int(nao_empresa.sum())
    nula = d["situacao"] == 1
    descartes["inscrição nula"] += int((nula & ~nao_empresa).sum())
    d = d[~nao_empresa & ~nula].copy()

    d["anos_de_vida"] = (d["data_situacao"] - d["data_abertura"]) // 10000
    d["ano_situacao"] = d["data_situacao"] // 10000
    ativa = d["situacao"] == 2
    invalida = ~ativa & ~(d["data_situacao"] >= d["data_abertura"])
    descartes["data de situação inválida"] += int(invalida.sum())

    # Para empresas ativas, a data da situação não interessa
    d.loc[ativa, ["anos_de_vida", "ano_situacao"]] = -1
    d = d[~invalida]
    descartes["empresas usadas"] += len(d)
    return d[CHAVES].astype(int).value_counts().rename("empresas").reset_index()


def processar(parte):
    zip_path = baixar(f"Estabelecimentos{parte}.zip")
    descartes = dict.fromkeys(
        ["linhas lidas", "campos essenciais ausentes", "não são empresas comuns",
         "inscrição nula", "data de situação inválida", "empresas usadas"], 0)
    resumos = []
    with pd.read_csv(
        zip_path, compression="zip", sep=";", header=None, encoding="latin-1",
        usecols=list(COLUNAS), dtype=str, chunksize=1_000_000, on_bad_lines="warn",
    ) as leitor:
        for pedaco in leitor:
            resumos.append(resumir(pedaco, descartes))
    total = pd.concat(resumos).groupby(CHAVES, as_index=False)["empresas"].sum()
    total.to_csv(PASTA_ZIP / f"resumo_{MES}_{parte}.csv", index=False)
    pd.Series(descartes).to_csv(PASTA_ZIP / f"descartes_{MES}_{parte}.csv", header=False)
    zip_path.unlink()


for parte in PARTES:
    if (PASTA_ZIP / f"descartes_{MES}_{parte}.csv").exists():
        print(f"Parte {parte}: já processada")
        continue
    print(f"Parte {parte}: baixando e processando...")
    processar(parte)

dados = pd.concat([pd.read_csv(PASTA_ZIP / f"resumo_{MES}_{p}.csv") for p in PARTES])
dados = dados.groupby(CHAVES, as_index=False)["empresas"].sum()
dados.to_csv(PASTA_SAIDA / "resumo_cnpj.csv", index=False)

pd.concat([
    pd.read_csv(PASTA_ZIP / f"descartes_{MES}_{p}.csv", header=None, names=["etapa", "n"])
    for p in PARTES
]).groupby("etapa", sort=False)["n"].sum()

## 4. Calcular a sobrevivência

Uma empresa "sobreviveu *k* anos" se não fechou antes do seu *k*-ésimo aniversário. Só calculamos células em que **todas** as empresas da coorte já tiveram tempo de completar *k* anos. As outras ficam vazias, e isso evita o viés de empresas novas parecerem sobreviver mais.

In [ ]:
SECOES = {
    "Agropecuária": (1, 3), "Indústria extrativa": (5, 9), "Indústria": (10, 33),
    "Eletricidade e gás": (35, 35), "Água e resíduos": (36, 39), "Construção": (41, 43),
    "Comércio": (45, 47), "Transporte e entregas": (49, 53),
    "Alojamento e alimentação": (55, 56), "Informação e comunicação": (58, 63),
    "Finanças e seguros": (64, 66), "Imobiliárias": (68, 68),
    "Serviços profissionais": (69, 75), "Serviços administrativos": (77, 82),
    "Educação": (85, 85), "Saúde": (86, 88), "Artes e esporte": (90, 93),
    "Outros serviços pessoais": (95, 96),
}


def secao(divisao):
    for nome, (ini, fim) in SECOES.items():
        if ini <= divisao <= fim:
            return nome
    return "Outros"


dados["setor"] = dados["divisao"].map({d: secao(d) for d in dados["divisao"].unique()})


def sobrevivencia(d, k, fechadas=FECHADAS):
    """% de empresas do grupo que não fecharam antes de completar k anos."""
    n = d["empresas"].sum()
    if n == 0:
        return np.nan
    fechou_antes = d["situacao"].isin(fechadas) & (d["anos_de_vida"] < k)
    return 100 * (1 - d.loc[fechou_antes, "empresas"].sum() / n)


def tabela_coortes(d, fechadas=FECHADAS):
    t = {}
    for ano, grupo in d.groupby("ano_abertura"):
        t[ano] = {k: sobrevivencia(grupo, k, fechadas)
                  for k in range(1, ULTIMO_ANO_COMPLETO - ano + 1)}
    return pd.DataFrame(t).T.sort_index(axis=0).sort_index(axis=1)


coortes = tabela_coortes(dados)
coortes.round(1).to_csv(PASTA_SAIDA / "coortes.csv")
coortes.round(0)

## 5. Conferências antes de concluir qualquer coisa

1. **Fechamentos em massa:** um ano muito acima dos vizinhos pode ser uma limpeza do cadastro feita pela Receita, e não uma crise.
2. **Definição de "fechou":** se a conclusão só aparece com uma das definições, ela é frágil.

In [ ]:
fechamentos = dados[dados["situacao"].isin([4, 8])]
print("Empresas por ano em que ficaram inaptas (4) ou baixadas (8):")
display(fechamentos.pivot_table(index="ano_situacao", columns="situacao",
                                values="empresas", aggfunc="sum", fill_value=0))

print("Sobrevivência após 3 anos (%), por definição de fechamento:")
display(pd.DataFrame({
    "inapta ou baixada": tabela_coortes(dados, {4, 8})[3],
    "só baixada": tabela_coortes(dados, {8})[3],
}).dropna().round(1))

## 6. Gráfico A: mapa de calor das coortes

In [ ]:
# Troque os títulos pela SUA conclusão depois de olhar os dados
TITULO_A = "Sobrevivência das empresas por ano de abertura"
TITULO_B = "Sobrevivência de 3 anos por setor"

AZUIS = LinearSegmentedColormap.from_list(
    "azuis", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
FUNDO, TINTA, TINTA2, GRADE = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"figure.facecolor": FUNDO, "axes.facecolor": FUNDO, "font.size": 11,
                     "text.color": TINTA, "axes.labelcolor": TINTA2,
                     "xtick.color": TINTA2, "ytick.color": TINTA2})

AMOSTRA = "" if sorted(PARTES) == list(range(10)) else "AMOSTRA, NÃO USAR NA ENTREGA. "
DEFINICAO = " ou ".join({3: "suspensas", 4: "inaptas", 8: "baixadas"}[s] for s in sorted(FECHADAS))
RODAPE = (f"{AMOSTRA}Fonte: Receita Federal, CNPJ ({MES}). Só matrizes. "
          f"Fechadas: empresas {DEFINICAO}.")


def cabecalho(fig, titulo, subtitulo):
    fig.text(0.02, 0.97, titulo, fontsize=17, fontweight="bold", va="top")
    fig.text(0.02, 0.92, subtitulo, fontsize=11.5, color=TINTA2, va="top")
    fig.text(0.02, 0.015, RODAPE, fontsize=9, color=TINTA2)


def grafico_coortes(t):
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.subplots_adjust(left=0.08, right=0.9, top=0.83, bottom=0.12)
    vmin, vmax = np.nanmin(t.values), np.nanmax(t.values)
    ax.pcolormesh(np.ma.masked_invalid(t.values), cmap=AZUIS, vmin=vmin, vmax=vmax,
                  edgecolors=FUNDO, linewidth=2)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.iat[i, j]
            if not np.isnan(v):
                escuro = (v - vmin) / max(vmax - vmin, 1e-9) > 0.5
                ax.text(j + 0.5, i + 0.5, f"{v:.0f}", ha="center", va="center",
                        fontsize=10, color="white" if escuro else TINTA)
    n = dados.groupby("ano_abertura")["empresas"].sum()
    for i, ano in enumerate(t.index):
        ax.text(t.shape[1] + 0.25, i + 0.5, f"{n[ano] / 1000:,.0f} mil".replace(",", "."),
                va="center", fontsize=9.5, color=TINTA2)
    ax.text(t.shape[1] + 0.25, -0.25, "Abertas", fontsize=9.5, color=TINTA2)
    ax.set_xticks(np.arange(t.shape[1]) + 0.5, t.columns)
    ax.set_yticks(np.arange(t.shape[0]) + 0.5, t.index)
    ax.set_xlabel("Anos completos desde a abertura", labelpad=8)
    ax.set_ylabel("Ano de abertura", labelpad=8)
    ax.invert_yaxis()
    ax.tick_params(length=0)
    ax.spines[:].set_visible(False)
    cabecalho(fig, TITULO_A,
              "% de empresas que ainda não tinham fechado. Mais escuro = mais sobreviventes.")
    return fig


fig = grafico_coortes(coortes)
fig.savefig(PASTA_SAIDA / "vis-coortes.png", dpi=200)
plt.show()

## 7. Gráfico B: comparação por setor

Compara empresas abertas em 2013–2015 (completaram 3 anos até 2018, antes da pandemia) com as abertas em 2020–2022 (até 2025).

In [ ]:
ANTES, DEPOIS, K = (2013, 2015), (2020, 2022), 3
MINIMO = 1000   # ignora setores com poucas empresas


def tabela_setores():
    linhas = []
    for setor, g in dados.groupby("setor"):
        a = g[g["ano_abertura"].between(*ANTES)]
        b = g[g["ano_abertura"].between(*DEPOIS)]
        if min(a["empresas"].sum(), b["empresas"].sum()) >= MINIMO:
            linhas.append((setor, sobrevivencia(a, K), sobrevivencia(b, K)))
    t = pd.DataFrame(linhas, columns=["setor", "antes", "depois"]).set_index("setor")
    return t.sort_values("depois")


def grafico_setores(t):
    fig, ax = plt.subplots(figsize=(11, 8))
    fig.subplots_adjust(left=0.24, right=0.95, top=0.82, bottom=0.13)
    y = np.arange(len(t))
    ax.hlines(y, t["antes"], t["depois"], color="#c3c2b7", linewidth=2, zorder=1)
    estilo = dict(s=95, edgecolor=FUNDO, linewidth=2, zorder=2)
    ax.scatter(t["antes"], y, color="#86b6ef", label=f"Abertas em {ANTES[0]}–{ANTES[1]}", **estilo)
    ax.scatter(t["depois"], y, color="#184f95", label=f"Abertas em {DEPOIS[0]}–{DEPOIS[1]}", **estilo)
    for i, (a, b) in enumerate(t[["antes", "depois"]].itertuples(index=False)):
        direita = b >= a
        ax.annotate(f"{b:.0f}", (b, i), xytext=(9 if direita else -9, 0),
                    textcoords="offset points", va="center",
                    ha="left" if direita else "right", fontsize=9.5)
    ax.set_yticks(y, t.index)
    ax.set_xlabel(f"% de empresas que sobreviveram {K} anos", labelpad=8)
    ax.margins(x=0.08, y=0.04)
    ax.grid(axis="x", color=GRADE, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
    ax.spines[:].set_visible(False)
    ax.legend(loc="lower left", bbox_to_anchor=(-0.01, 1.0), ncols=2, frameon=False)
    cabecalho(fig, TITULO_B,
              f"% de empresas que não tinham fechado após {K} anos, em dois períodos de abertura.")
    return fig


setores = tabela_setores()
setores.round(1).to_csv(PASTA_SAIDA / "setores.csv")
fig = grafico_setores(setores)
fig.savefig(PASTA_SAIDA / "vis-setores.png", dpi=200)
plt.show()
setores.assign(diferenca=setores["depois"] - setores["antes"]).round(1)

## 8. Salvar os resultados

No Colab, esta célula baixa um zip com as tabelas e figuras. Coloque a pasta `dados/` no repositório junto com este notebook.

In [ ]:
try:
    import shutil

    from google.colab import files
    files.download(shutil.make_archive("resultados", "zip", ".", "dados"))
except ImportError:
    print("Resultados em", PASTA_SAIDA.resolve())